/content/drive/MyDrive/research

In [1]:
%pip install thop opencv-python psutil requests matplotlib numpy huggingface_hub

In [ ]:
# --- Imports ---
import sys
sys.path.append('/content/drive/MyDrive/research')

from ultralytics import YOLO
import os
os.chdir('/content/drive/MyDrive/research')
#from drive.MyDrive.research.ultralytics import YOLO
import csv

RESEARCH_DIR="custom_experiments"
# ---------------- CONFIG ----------------
DATA_YAML_100 = "TACO_splits_with_test/Custom_TACO_split_100/data.yaml"
DATA_YAML_80  = "TACO_splits_with_test/Custom_TACO_split_80/data.yaml"
DATA_YAML_50  = "TACO_splits_with_test/Custom_TACO_split_50/data.yaml"
DATA_YAML_30  = "TACO_splits_with_test/Custom_TACO_split_30/data.yaml"

DATA_YAMLS = {
    "100": DATA_YAML_100,
    "80": DATA_YAML_80,
    "50": DATA_YAML_50,
    "30": DATA_YAML_30,
}

OUTPUT_CSV = "evaluation_results.csv"
DEVICE = 0  # use 0 for GPU, "cpu" for CPU

MODEL_N = "yolov12n.yaml"
MODEL_S = "yolov12s.yaml"
MODEL_M = "yolov12m.yaml"
MODEL_L = "yolov12l.yaml"

MODELS = [MODEL_N, MODEL_S, MODEL_M, MODEL_L]
MDLS = ["N", "S", "M", "L"]
# ----------------------------------------

def get_split_from_experiment(exp_name):
    """
    Extracts dataset split (100, 80, 50, 30) from experiment name.
    Example: exp_something_80 -> "80"
    """
    parts = exp_name.split("_")

    for part in reversed(parts):
        if part in DATA_YAMLS:
            return part

    return None

def find_models(base_dir):
    models = []

    for model_size in MDLS:
        size_path = os.path.join(base_dir, model_size)

        if not os.path.exists(size_path):
            continue

        for exp in os.listdir(size_path):
            exp_path = os.path.join(size_path, exp)

            weights_path = os.path.join(exp_path, "train", "weights", "last.pt")
            print(f"checking for model: {weights_path}")
            if os.path.exists(weights_path):

                models.append({
                    "model_size": model_size,
                    "experiment": exp,
                    "weights": weights_path
                })

    return models


def evaluate_model(model_info):
    exp_name = model_info["experiment"]
    print(f"Evaluating: {model_info['model_size']} | {exp_name}")

    split = get_split_from_experiment(exp_name)

    if split is None:
        print(f"Skipping {exp_name}: No valid split found")
        return None

    data_yaml = DATA_YAMLS[split]

    try:
        model = YOLO(model_info["weights"])

        results = model.val(
            data=data_yaml,
            split="test",
            device=DEVICE,
            verbose=False,
            project="/content/inference/"+model_info["model_size"]+"/"+exp_name
        )

        metrics = results.results_dict

        return {
            "model_size": model_info["model_size"],
            "experiment": exp_name,
            "mAP50": metrics.get("metrics/mAP50(B)", None),
            "mAP50-95": metrics.get("metrics/mAP50-95(B)", None),
            "precision": metrics.get("metrics/precision(B)", None),
            "recall": metrics.get("metrics/recall(B)", None)
        }

    except Exception as e:
        print(f"Error evaluating {exp_name}: {e}")
        return None


def main():
    models = find_models(RESEARCH_DIR)

    print(f"Found {len(models)} models.")

    results = []

    for model_info in models:
        res = evaluate_model(model_info)
        if res:
            results.append(res)

    # Save to CSV
    keys = ["model_size", "experiment", "mAP50", "mAP50-95", "precision", "recall"]

    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(results)

    print(f"\nResults saved to {OUTPUT_CSV}")


if __name__ == '__main__':
    main()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/yolov12/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
checking for model: custom_experiments/N/EX_5_F1_A1_N10_100/train/weights/last.pt
checking for model: custom_experiments/N/EX_5_F1_A1_N30_100/train/weights/last.pt
checking for model: custom_experiments/N/EX_5_F1_A1_N80_100/train/weights/last.pt
checking for model: custom_experiments/N/EX_5_F1_A1_N10_80/train/weights/last.pt
checking for model: custom_experiments/N/EX_5_F1_A1_N30_80/train/weights/last.pt
checking for model: custom_experiments/N/EX_5_F1_A1_N80_80/train/weights/last.pt
checking for model: custom_experiments/N/EX_5_F1_A1_N10_50/train/weights/last.pt
checking for model: custom_expe

100%|██████████| 755k/755k [00:00<00:00, 129MB/s]
val: Scanning /content/drive/MyDrive/research/TACO_splits_with_test/Custom_TACO_split_100/labels/test.cache... 41 images, 0 backgrounds, 0 corrupt: 100%|██████████| 41/41 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:18<00:00,  6.27s/it]


                   all         41         84      0.298      0.327      0.317      0.243
Speed: 2.7ms preprocess, 14.6ms inference, 0.0ms loss, 7.5ms postprocess per image
Results saved to /content/inference/N/EX_5_F1_A1_N10_100/val
Evaluating: N | EX_5_F1_A1_N30_100
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv12n summary (fused): 376 layers, 2,508,929 parameters, 0 gradients, 5.8 GFLOPs


val: Scanning /content/drive/MyDrive/research/TACO_splits_with_test/Custom_TACO_split_100/labels/test.cache... 41 images, 0 backgrounds, 0 corrupt: 100%|██████████| 41/41 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:02<00:00,  1.37it/s]


                   all         41         84      0.454      0.428      0.428      0.345
Speed: 1.9ms preprocess, 4.3ms inference, 0.0ms loss, 3.7ms postprocess per image
Results saved to /content/inference/N/EX_5_F1_A1_N30_100/val
Evaluating: N | EX_5_F1_A1_N80_100
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv12n summary (fused): 376 layers, 2,508,929 parameters, 0 gradients, 5.8 GFLOPs


...

                   all         32         37      0.656      0.756      0.698      0.561
Speed: 2.8ms preprocess, 23.3ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to /content/inference/L/EX_5_F1_A1_L50_80/val
Evaluating: L | EX_5_F1_A1_L50_50
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv12l summary (fused): 674 layers, 26,395,865 parameters, 0 gradients, 82.1 GFLOPs


val: Scanning /content/drive/MyDrive/research/TACO_splits_with_test/Custom_TACO_split_50/labels/test.cache... 20 images, 0 backgrounds, 0 corrupt: 100%|██████████| 20/20 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:02<00:00,  1.16s/it]


                   all         20         22      0.765      0.781      0.893      0.801
Speed: 4.4ms preprocess, 26.0ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /content/inference/L/EX_5_F1_A1_L50_50/val
Evaluating: L | EX_5_F1_A1_L50_30
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv12l summary (fused): 674 layers, 26,395,865 parameters, 0 gradients, 82.1 GFLOPs


val: Scanning /content/drive/MyDrive/research/TACO_splits_with_test/Custom_TACO_split_30/labels/test.cache... 12 images, 0 backgrounds, 0 corrupt: 100%|██████████| 12/12 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:01<00:00,  1.53s/it]


                   all         12         12      0.803      0.848        0.9      0.822
Speed: 0.2ms preprocess, 27.5ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /content/inference/L/EX_5_F1_A1_L50_30/val

Results saved to evaluation_results.csv


In [ ]:
!cp -r /content/inference/ /content/drive/MyDrive/research/custom_experiments/


In [ ]:
import os
import csv
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/research/custom_experiments/inference"
OUTPUT_CSV = "/content/drive/MyDrive/research/final_results.csv"

DATA_SPLITS = ["100", "80", "50", "30"]

def get_split_from_experiment(exp_name):
    for part in exp_name.split("_"):
        if part in DATA_SPLITS:
            return part
    return None


def extract_metrics_from_csv(csv_path):
    df = pd.read_csv(csv_path)

    # Take last row (final epoch)
    last = df.iloc[-1]

    return {
        "mAP50": last.get("metrics/mAP50(B)", None),
        "mAP50-95": last.get("metrics/mAP50-95(B)", None),
        "precision": last.get("metrics/precision(B)", None),
        "recall": last.get("metrics/recall(B)", None),
    }


results = []

for model_size in ["N", "S", "M", "L"]:
    model_path = os.path.join(BASE_DIR, model_size)

    if not os.path.exists(model_path):
        continue

    for exp in os.listdir(model_path):
        exp_path = os.path.join(model_path, exp)

        # Try common locations
        possible_paths = [
            os.path.join(exp_path, "train", "results.csv"),
            os.path.join(exp_path, "val", "results.csv"),
            os.path.join(exp_path, "results.csv"),
        ]

        csv_file = None
        for p in possible_paths:
            if os.path.exists(p):
                csv_file = p
                break

        if csv_file is None:
            print(f"Skipping {exp} (no results.csv found)")
            continue

        split = get_split_from_experiment(exp)

        metrics = extract_metrics_from_csv(csv_file)

        results.append({
            "model_size": model_size,
            "experiment": exp,
            "split": split,
            **metrics
        })

# Save final CSV
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Aggregated results saved to: {OUTPUT_CSV}")
print(df.head())

Skipping EX_5_F1_A1_N50_100 (no results.csv found)
Skipping EX_5_F1_A1_N50_50 (no results.csv found)
Skipping EX_5_F1_A1_N50_30 (no results.csv found)
Skipping EX_5_F1_A1_N50_80 (no results.csv found)
Skipping EX_5_F1_A1_N10_100 (no results.csv found)
Skipping EX_5_F1_A1_N30_100 (no results.csv found)
Skipping EX_5_F1_A1_N80_100 (no results.csv found)
Skipping EX_5_F1_A1_N10_80 (no results.csv found)
Skipping EX_5_F1_A1_N30_80 (no results.csv found)
Skipping EX_5_F1_A1_N80_80 (no results.csv found)
Skipping EX_5_F1_A1_N10_50 (no results.csv found)
Skipping EX_5_F1_A1_N30_50 (no results.csv found)
Skipping EX_5_F1_A1_N80_50 (no results.csv found)
Skipping EX_5_F1_A1_N10_30 (no results.csv found)
Skipping EX_5_F1_A1_N30_30 (no results.csv found)
Skipping EX_5_F1_A1_N80_30 (no results.csv found)
Skipping EX_5_F1_A1_S50_100 (no results.csv found)
Skipping EX_5_F1_A1_S50_30 (no results.csv found)
Skipping EX_5_F1_A1_S50_80 (no results.csv found)
Skipping EX_5_F1_A1_S50_50 (no results.csv fo

In [2]:
import sys
sys.path.append('/content/drive/MyDrive/research')

from ultralytics import YOLO
import os
import numpy as np

os.chdir('/content/drive/MyDrive/research')

# ---------------- CONFIG ----------------
VARS = ["1", "2", "3", "4"]  # <-- YOUR FOLDS

DATA_TEMPLATE = "TACO_5_variations/var{VAR}/subset_{SIZE}/data.yaml"
DATA_SIZES = ["100", "80", "50", "30"]

MDLS = ["N", "S", "M", "L"]
PRETR_NOTATIONS = ["10", "30", "50", "80"]

EXP = "8"

BASE_PATH = "/content/drive/MyDrive/research/variative_cust_experiments"

# ----------------------------------------

def evaluate_model(weights_path, data_yaml):
    model = YOLO(weights_path)

    metrics = model.val(
        data=data_yaml,
        split="test",
        imgsz=640,
        conf=0.05,
        verbose=False
    )

    return {
        "map50": metrics.box.map50,
        "map50_95": metrics.box.map,
        "precision": metrics.box.mp,
        "recall": metrics.box.mr
    }


def main():
    final_results = {}

    for mdl in MDLS:
        final_results[mdl] = {}

        for size in DATA_SIZES:
            final_results[mdl][size] = {}

            for p in PRETR_NOTATIONS:

                fold_metrics = []

                for VAR in VARS:
                    data_yaml = DATA_TEMPLATE.format(VAR=VAR, SIZE=size)

                    weights_path = os.path.join(
                        BASE_PATH,
                        f"var{VAR}_results",
                        mdl,
                        f"EX_{EXP}_F1_A1_{mdl}{p}_{size}",  # ⚠️ adjust if needed
                        "train",
                        "weights",
                        "best.pt"
                    )

                    if not os.path.exists(weights_path):
                        print(f"Missing: {weights_path}")
                        continue

                    metrics = evaluate_model(weights_path, data_yaml)
                    fold_metrics.append(metrics)

                if len(fold_metrics) == 0:
                    continue

                stats = {}
                for key in fold_metrics[0]:
                    values = [m[key] for m in fold_metrics]

                    stats[key] = {
                        "mean": float(np.mean(values)),
                        "std": float(np.std(values))
                    }

                final_results[mdl][size][p] = stats

                print(f"\nModel {mdl} | Data {size} | Pretrain {p}")
                for metric in stats:
                    print(f"{metric}: {stats[metric]['mean']:.4f} ± {stats[metric]['std']:.4f}")

    print("\n==== FINAL RESULTS ====")
    for mdl in final_results:
        for size in final_results[mdl]:
            for p in final_results[mdl][size]:
                print(f"{mdl} | {size} | {p} -> {final_results[mdl][size][p]}")


if __name__ == "__main__":
    main()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/yolov12/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
Missing: /content/drive/MyDrive/research/variative_cust_experiments/var1_results/N/EX_8_F1_A1_N10_100/train/weights/best.pt
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12n summary (fused): 376 layers, 2,508,929 parameters, 0 gradients, 5.8 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 45.5MB/s]
val: Scanning /content/drive/MyDrive/research/TACO_5_variations/var2/subset_100/labels/test... 41 images, 0 backgrounds, 0 corrupt: 100%|██████████| 41/41 [00:10<00:00,  4.07it/s]


val: New cache created: /content/drive/MyDrive/research/TACO_5_variations/var2/subset_100/labels/test.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:08<00:00,  2.87s/it]


                   all         41         46      0.546      0.474      0.545      0.449
Speed: 1.3ms preprocess, 13.8ms inference, 0.0ms loss, 6.3ms postprocess per image
Results saved to runs/detect/val
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12n summary (fused): 376 layers, 2,508,929 parameters, 0 gradients, 5.8 GFLOPs


...

                   all         13         17      0.786      0.677      0.783      0.693
Speed: 0.2ms preprocess, 8.2ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to runs/detect/val190
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12l summary (fused): 674 layers, 26,395,865 parameters, 0 gradients, 82.1 GFLOPs


val: Scanning /content/drive/MyDrive/research/TACO_5_variations/var3/subset_30/labels/test.cache... 13 images, 0 backgrounds, 0 corrupt: 100%|██████████| 13/13 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:01<00:00,  1.51s/it]


                   all         13         24      0.868      0.924      0.915      0.805
Speed: 0.2ms preprocess, 8.1ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/val191
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12l summary (fused): 674 layers, 26,395,865 parameters, 0 gradients, 82.1 GFLOPs


val: Scanning /content/drive/MyDrive/research/TACO_5_variations/var4/subset_30/labels/test.cache... 13 images, 0 backgrounds, 0 corrupt: 100%|██████████| 13/13 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:01<00:00,  1.54s/it]


                   all         13         19      0.932      0.776      0.894      0.769
Speed: 0.2ms preprocess, 8.2ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs/detect/val192

Model L | Data 30 | Pretrain 80
map50: 0.8643 ± 0.0581
map50_95: 0.7559 ± 0.0467
precision: 0.8618 ± 0.0597
recall: 0.7923 ± 0.1015

==== FINAL RESULTS ====
N | 100 | 10 -> {'map50': {'mean': 0.4795316331283392, 'std': 0.0757318052705588}, 'map50_95': {'mean': 0.38843200012416124, 'std': 0.06320054591927397}, 'precision': {'mean': 0.6289917765772548, 'std': 0.2001224550595137}, 'recall': {'mean': 0.3559506823268473, 'std': 0.16137842826217103}}
N | 100 | 30 -> {'map50': {'mean': 0.3332702233196148, 'std': 0.13464364337012052}, 'map50_95': {'mean': 0.2822863667087798, 'std': 0.09758434492709395}, 'precision': {'mean': 0.45098039215686275, 'std': 0.0835732800819897}, 'recall': {'mean': 0.23130511463844794, 'std': 0.23188091849468945}}
N | 100 | 50 -> {'map50': {'mean': 0.449128957431

In [2]:
import sys
sys.path.append('/content/drive/MyDrive/research')

from ultralytics import YOLO
import os
import numpy as np
import pandas as pd

os.chdir('/content/drive/MyDrive/research')

# ---------------- CONFIG ----------------
VARS = ["1", "2", "3", "4"]

DATA_TEMPLATE = "TACO_5_variations/var{VAR}/subset_{SIZE}/data.yaml"
DATA_SIZES = ["100", "80", "50", "30"]

MDLS = ["N", "S", "M", "L"]

EXP = "9"

BASE_PATH = "/content/drive/MyDrive/research/variative_synt_experiments"

OUTPUT_CSV = "/content/yolo_variative_results.csv"
# ----------------------------------------


def evaluate_model(weights_path, data_yaml):
    model = YOLO(weights_path)

    metrics = model.val(
        data=data_yaml,
        split="test",
        imgsz=640,
        conf=0.05,
        verbose=False
    )

    return {
        "map50": metrics.box.map50,
        "map50_95": metrics.box.map,
        "precision": metrics.box.mp,
        "recall": metrics.box.mr
    }


def main():
    rows = []

    for mdl in MDLS:

        for size in DATA_SIZES:
            fold_metrics = []

            for VAR in VARS:
                data_yaml = DATA_TEMPLATE.format(VAR=VAR, SIZE=size)

                weights_path = os.path.join(
                    BASE_PATH,
                    f"var{VAR}_results",
                    mdl,
                    f"EX_{EXP}_F1_A1_{mdl}_{size}",
                    "train",
                    "weights",
                    "best.pt"
                )

                if not os.path.exists(weights_path):
                    print(f"Missing: {weights_path}")
                    continue

                metrics = evaluate_model(weights_path, data_yaml)
                fold_metrics.append(metrics)

            if len(fold_metrics) == 0:
                continue

            # aggregate across folds
            stats = {}
            for key in fold_metrics[0]:
                values = [m[key] for m in fold_metrics]
                stats[key] = {
                    "mean": float(np.mean(values)),
                    "std": float(np.std(values))
                }

            print(f"\nModel {mdl} | Data {size}")
            for metric in stats:
                print(f"{metric}: {stats[metric]['mean']:.4f} ± {stats[metric]['std']:.4f}")

            # store row for CSV
            row = {
                "model": mdl,
                "size": size
            }

            for metric in stats:
                row[f"{metric}_mean"] = stats[metric]["mean"]
                row[f"{metric}_std"] = stats[metric]["std"]

            rows.append(row)

    # ---------------- SAVE CSV ----------------
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False)

    print("\n==== FINAL RESULTS SAVED ====")
    print(f"CSV saved to: {OUTPUT_CSV}")
    print(df)


if __name__ == "__main__":
    main()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/yolov12/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12n summary (fused): 376 layers, 2,508,929 parameters, 0 gradients, 5.8 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 170MB/s]
val: Scanning /content/drive/MyDrive/research/TACO_5_variations/var1/subset_100/labels/test... 41 images, 0 backgrounds, 0 corrupt: 100%|██████████| 41/41 [00:15<00:00,  2.70it/s]

val: New cache created: /content/drive/MyDrive/research/TACO_5_variations/var1/subset_100/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:14<00:00,  4.86s/it]


                   all         41         63      0.443      0.328      0.311      0.215
Speed: 2.6ms preprocess, 15.6ms inference, 0.0ms loss, 6.2ms postprocess per image
Results saved to runs/detect/val
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12n summary (fused): 376 layers, 2,508,929 parameters, 0 gradients, 5.8 GFLOPs


...

                   all         14         16      0.909      0.773      0.876      0.822
Speed: 0.1ms preprocess, 8.2ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/val46
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12l summary (fused): 674 layers, 26,395,865 parameters, 0 gradients, 82.1 GFLOPs


val: Scanning /content/drive/MyDrive/research/TACO_5_variations/var2/subset_30/labels/test.cache... 13 images, 0 backgrounds, 0 corrupt: 100%|██████████| 13/13 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:01<00:00,  1.45s/it]


                   all         13         17      0.859      0.694      0.761      0.728
Speed: 0.1ms preprocess, 8.5ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to runs/detect/val47
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12l summary (fused): 674 layers, 26,395,865 parameters, 0 gradients, 82.1 GFLOPs


val: Scanning /content/drive/MyDrive/research/TACO_5_variations/var3/subset_30/labels/test.cache... 13 images, 0 backgrounds, 0 corrupt: 100%|██████████| 13/13 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:01<00:00,  1.44s/it]


                   all         13         24      0.483      0.714      0.685      0.582
Speed: 0.2ms preprocess, 8.1ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs/detect/val48
Missing: /content/drive/MyDrive/research/variative_synt_experiments/var4_results/L/EX_9_F1_A1_L_30/train/weights/best.pt

Model L | Data 30
map50: 0.7739 ± 0.0787
map50_95: 0.7104 ± 0.0986
precision: 0.7505 ± 0.1899
recall: 0.7271 ± 0.0332

==== FINAL RESULTS SAVED ====
CSV saved to: /content/yolo_variative_results.csv
   model size  map50_mean  map50_std  map50_95_mean  map50_95_std  \
0      N  100    0.353857   0.072341       0.258728      0.061859   
1      N   80    0.584256   0.057049       0.470660      0.072353   
2      N   50    0.584051   0.069512       0.492611      0.069356   
3      N   30    0.588057   0.147085       0.519406      0.133633   
4      S  100    0.369155   0.014446       0.283233      0.025225   
5      S   80    0.716563   0.017644       0.621438      0.